In [ ]:
# DIFT sliding-window extension configuration.
%reload_ext autoreload
%autoreload 2
%cd ~/erc-src/cuneiform-ocr-sign-alignment-worktree
%env PATH=$HOME/.local/bin:$PATH

import os

from dotenv import load_dotenv

import sign_alignment.pipeline as pp
import sign_alignment.pipeline_2 as pp2
from sign_alignment.data_source import EBLMongoCanonicalSource, LocalDataSource
from sign_alignment.detector import ModelConfig, TabletImageDetector
from sign_alignment.dift_align import DiftAlignmentConfig, DiftRuntime
from sign_alignment.visualizer import ColorConfig

ANNOTATIONS_DIR = os.path.expanduser(
    "~/erc-work-data/data-of-cuneiform-ocr-data/filtered_annotations"
)
CONFIG_FILE = "configs/detr.py"
CHECKPOINT_FILE = os.path.expanduser(
    "~/erc-work-data/retrained_models/detr-173/epoch_1000.pth"
)
DIFT_CHECKPOINT = os.path.expanduser("~/erc-src/ProtoSnap/weights/SD_with_prompt")
CANONICAL_FEATURE_DIR = os.path.expanduser(
    "~/erc-work-data/signs_alignment_data/precompute_feautures"
)
SCORE_THRESHOLD = 0.5
OUTPUT_DIR = "alignment_results_2"
SAMPLE_NAME = "YBC.12860"
CROP_INDEX = 5

load_dotenv()
MONGODB_URI = os.getenv("MONGODB_URI")
if not MONGODB_URI:
    raise ValueError("MONGODB_URI is required for canonical sign images")


In [ ]:
# Initialize the base context; only the coarse-alignment step comes from pipeline_2.
model_config = ModelConfig(
    config_file=CONFIG_FILE,
    checkpoint_file=CHECKPOINT_FILE,
    device="auto",
)
if "tablet_detector" not in globals() or getattr(tablet_detector, "model", None) is None:
    tablet_detector = TabletImageDetector(
        default_score_threshold=SCORE_THRESHOLD,
        model_config=model_config,
        keep_crops=True,
        is_crop_itself=False,
    )
else:
    print("Reusing existing tablet_detector instance.")

dift = DiftRuntime(
    checkpoint=DIFT_CHECKPOINT,
    feature_dir=CANONICAL_FEATURE_DIR,
    config=DiftAlignmentConfig(affine_probe_padding_ratio=0.1),
)
context = pp.CropContext(
    tablet_detector=tablet_detector,
    local_source=LocalDataSource(ANNOTATIONS_DIR),
    color_config=ColorConfig,
    output_dir=OUTPUT_DIR,
    img_idx=CROP_INDEX,
    dift=dift,
    sign_source=EBLMongoCanonicalSource(MONGODB_URI),
)
feature_config = pp2.FeatureCoarseAlignmentConfig(
    step_px=100,
    search_margin_px=100,
    assignment_min_score=0.0,
)
runner = pp.Runner(context, pp.VisOptions(info=True, display=True, save=True))


In [ ]:
# Load the sample, detect signs, and produce the base row/sign matches.
runner.choose_sample(name=SAMPLE_NAME)
runner.run([
    pp.Step("Load data", pp.load_data, pp.vis_loaded_data),
    pp.Step("Detect signs", pp.detect_signs, pp.vis_detections),
    pp.Step("Transform GT to crop", pp.transform_gt_to_crop, pp.vis_crop_ground_truth),
    pp.Step("Detection statistics", lambda _: None, pp.vis_detection_statistics),
    pp.Step("Create box sets", pp.create_box_sets, pp.vis_box_sets),
    pp.Step("Detect rows", pp.detect_rows, pp.vis_detected_rows_info),
    pp.Step("Match rows", pp.match_rows, pp.vis_row_matches),
    pp.Step("Visualize rows", lambda _: None, pp.vis_detection_rows),
    pp.Step("Match signs", pp.match_signs_in_rows, pp.vis_sign_matches),
])


In [ ]:
# Run the extension in place of pp.align_text_rows.
runner.run([
    pp.Step("Unload detector", pp.unload_detector),
    pp.Step("Setup source signs", pp.setup_source_signs, pp.vis_source_signs),
    pp.Step(
        "DIFT sliding-window coarse alignment",
        lambda ctx: pp2.align_text_rows_with_feature_search(ctx, feature_config),
        pp2.vis_feature_coarse_alignment,
    ),
])


In [ ]:
# Inspect selected DIFT assignments.
feature_run = pp2.get_feature_coarse_run(context)
feature_assignments = [
    {
        "text_row": row.text_row_idx,
        "det_row": row.det_row_idx,
        "text_idx": item.text_idx,
        "sign": item.sign_name,
        "score": item.score,
        "geometry": item.geometry,
        "support": item.support,
        "inliers": f"{item.n_inliers}/{item.n_matches}",
        "box": tuple(map(int, item.box.crop_bounds())),
    }
    for row in feature_run.rows.values()
    for item in row.assignments
    if item.source == "feature"
]
sorted(feature_assignments, key=lambda item: item["score"], reverse=True)


In [ ]:
# Reuse the base diagnostics and PSR downstream flow.
runner.run([
    pp.Step("Build sign match info", pp.build_sign_match_info, pp.vis_sign_match_info),
    pp.Step("Offset analysis", lambda _: None, pp.vis_offset_analysis),
    pp.Step("Create PSR optimizer", pp.create_psr_optimizer, pp.vis_psr_optimizer),
    pp.Step("Optimize until DIFT probe", pp.optimize_psr_until_dift_probe, pp.vis_optimization),
    pp.Step("Source sign overlay", pp.create_source_sign_overlay, pp.vis_source_sign_overlay),
    pp.Step("DIFT affine probe", pp.run_dift_affine_probe, pp.vis_dift_affine_probe),
    pp.Step("Finish PSR optimization", pp.optimize_psr_after_dift_probe, pp.vis_optimization),
])


In [ ]:
# Inspect final optimization results.
runner.run([
    pp.Step("Loss history", lambda _: None, pp.vis_loss_history),
    pp.Step("Results comparison", lambda _: None, pp.vis_results_comparison),
    pp.Step("Parameter changes", lambda _: None, pp.vis_parameter_changes),
])
